# Cleaning Messy IMDB Dataset

In [55]:
import numpy as np
import pandas as pd
from pathlib import Path
import datetime
import logging
import importlib
import chardet

import library

importlib.reload(library)

log = library.logger.getLogger(__name__)
log.setLevel(logging.DEBUG)

In [56]:
library.tools.set_display_options()

## Get the path to the CSV dataset

In [57]:
path = Path().cwd().joinpath('imdb', 'messy_imdb_dataset.csv')

Get the encoding of the CSV file (non UTF-8)

In [58]:
encoding = library.path.detect_encoding(path)
log.debug(f'Detecting Encoding {encoding}')

Detecting Encoding Windows-1252


Read the CSV into a Pandas Dataframe

In [59]:
df = pd.read_csv(path, sep=';', encoding=encoding)
df

,IMBD title ID,Original titlÊ,Release year,Genrë¨,Duration,Country,Content Rating,Director,Unnamed: 8,Income,Votes,Score
0,tt0111161,The Shawshank Redemption,1995-02-10,Drama,142,USA,R,Frank Darabont,NaN,$ 28815245,2.278.845,9.3
1,tt0068646,The Godfather,09 21 1972,"Crime, Drama",175,USA,R,Francis Ford Coppola,NaN,$ 246120974,1.572.674,9.2
2,tt0468569,The Dark Knight,23 -07-2008,"Action, Crime, Drama",152,US,PG-13,Christopher Nolan,NaN,$ 1005455211,2.241.615,9.
3,tt0071562,The Godfather: Part II,1975-09-25,"Crime, Drama",220,USA,R,Francis Ford Coppola,NaN,"$ 4o8,035,783",1.098.714,"9,.0"
4,tt0110912,Pulp Fiction,1994-10-28,"Crime, Drama",,USA,R,Quentin Tarantino,NaN,$ 222831817,1.780.147,"8,9f"
5,tt0167260,The Lord of the Rings: The Return of the King,22 Feb 04,"Action, Adventure, Drama",201,New Zealand,PG-13,Peter Jackson,NaN,$ 1142271098,1.604.280,08.9
6,tt0108052,Schindler's List,1994-03-11,"Biography, Drama, History",Nan,USA,R,Steven Spielberg,NaN,$ 322287794,1.183.248,8.9
7,tt0050083,12 Angry Men,1957-09-04,"Crime, Drama",96,USA,Not Rated,Sidney Lumet,NaN,$ 576,668.473,8.9
8,tt1375666,Inception,2010-09-24,"Action, Adventure, Sci-Fi",148,USA,PG-13,Christopher Nolan,NaN,$ 869784991,2.002.816,8..8
9,tt0137523,Fight Club,10-29-99,Drama,Inf,UK,R,David Fincher,NaN,$ 101218804,1.807.440,8.8


## Getting the general info to see what we're working with

In [60]:
log.info(f'General Info {df.info()}')
log.info(f'General Info {df.shape}')

General Info None
General Info (101, 12)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   IMBD title ID   100 non-null    object 
 1   Original titlÊ  100 non-null    object 
 2   Release year    100 non-null    object 
 3   Genrë¨          100 non-null    object 
 4   Duration        99 non-null     object 
 5   Country         100 non-null    object 
 6   Content Rating  77 non-null     object 
 7   Director        100 non-null    object 
 8   Unnamed: 8      0 non-null      float64
 9   Income          100 non-null    object 
 10   Votes          100 non-null    object 
 11  Score           100 non-null    object 
dtypes: float64(1), object(11)
memory usage: 9.6+ KB


### Checking for null indices and rows

In [61]:
na_items = pd.isnull(df).sum()
log.debug(f'na_items {na_items}')
null_indices = library.tools.get_null_indices(df)
null_columns, null_rows = library.tools.get_all_nulls(df)
log.info(f'Null indices {len(df.columns)}/{len(null_indices)}\n{null_indices}')
log.info(f'Null columns {null_columns}')
log.info(f'Null rows {null_rows}')

na_items IMBD title ID       1
Original titlÊ      1
Release year        1
Genrë¨              1
Duration            2
Country             1
Content Rating     24
Director            1
Unnamed: 8        101
Income              1
 Votes              1
Score               1
dtype: int64
Null indices 12/12
{'IMBD title ID': Index([13], dtype='int64'), 'Original titlÊ': Index([13], dtype='int64'), 'Release year': Index([13], dtype='int64'), 'Genrë¨': Index([13], dtype='int64'), 'Duration': Index([13, 14], dtype='int64'), 'Country': Index([13], dtype='int64'), 'Content Rating': Index([ 13,  27,  28,  36,  40,  41,  47,  48,  56,  58,  62,  63,  65,  66,
        69,  70,  81,  86,  89,  90,  92,  93,  98, 100],
      dtype='int64'), 'Director': Index([13], dtype='int64'), 'Unnamed: 8': Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100],
      dtype='int64', length=101), 'Income': Index([13], dtype='int64'), ' Votes ': 

## Creating a copy of the data before modifying it

In [62]:
df_checkpoint1 = df.copy()

In [63]:
df_checkpoint1.shape

(101, 12)

### Row 13 and Column 'Unnamed: 8:' are null - dropping them

In [64]:
if not null_columns.empty:
    df_checkpoint1 = df_checkpoint1.drop(columns=null_columns)
if not null_rows.empty:
    df_checkpoint1 = df_checkpoint1.drop(index=null_rows)
# Comparing to make sure it's dropped
log.debug(f'{df.equals(df_checkpoint1)}: {df.shape}/{df_checkpoint1.shape}')

False: (101, 12)/(100, 11)


Ensuring we have no more null values

In [65]:
null_indices = library.tools.get_null_indices(df_checkpoint1)
log.debug(f'null_indices: {null_indices}')

null_indices: {'Duration': Index([14], dtype='int64'), 'Content Rating': Index([ 27,  28,  36,  40,  41,  47,  48,  56,  58,  62,  63,  65,  66,  69,
        70,  81,  86,  89,  90,  92,  93,  98, 100],
      dtype='int64')}


## Dropping the 'IMDB title ID' because we don't need it

In [66]:
df_checkpoint1 = df_checkpoint1.drop('IMBD title ID', axis=1)

# If I wanted to rename the misspelled column
# df_checkpoint1 = df_checkpoint1.rename({'IMBD title ID': 'IMDB title ID'}, axis=1)

In [67]:
df_checkpoint1.columns

Index(['Original titlÊ', 'Release year', 'Genrë¨', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Cleaning 'Original Title' column

In [68]:
df_checkpoint1 = df_checkpoint1.rename({'Original titlÊ': 'Original Title'}, axis=1)
df_checkpoint1.columns

Index(['Original Title', 'Release year', 'Genrë¨', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Converting 'Release year' column to DateTime

In [69]:
df_checkpoint1['Release year']

0                       1995-02-10
1                       09 21 1972
2                      23 -07-2008
3                       1975-09-25
4                       1994-10-28
5                        22 Feb 04
6                       1994-03-11
7                       1957-09-04
8                       2010-09-24
9                         10-29-99
10                      1994-10-06
11                      2002-01-18
12          23rd December of 1966 
14                      1999-05-07
15                        01/16-03
16                      1980-09-19
17                      1990-09-20
18                      18/11/1976
19                      2014-11-06
20                      1995-12-15
21                      1991-03-05
22                      1977-10-20
23                      1998-10-30
24                      2000-10-03
25                      2003-05-09
26                      2003-04-18
27                      1997-12-20
28                      2019-11-07
29                  

Convert to Timestamp

In [70]:
df_checkpoint1.loc[:, 'Release year'] = pd.to_datetime(df_checkpoint1.loc[:, 'Release year'], yearfirst=True, format='mixed', errors='coerce')
df_checkpoint1.loc[:, 'Release year']

0      1995-02-10 00:00:00
1      1972-09-21 00:00:00
2      2008-07-23 00:00:00
3      1975-09-25 00:00:00
4      1994-10-28 00:00:00
5      2022-02-04 00:00:00
6      1994-03-11 00:00:00
7      1957-09-04 00:00:00
8      2010-09-24 00:00:00
9      1999-10-29 00:00:00
10     1994-10-06 00:00:00
11     2002-01-18 00:00:00
12     1966-12-23 00:00:00
14     1999-05-07 00:00:00
15     2003-01-16 00:00:00
16     1980-09-19 00:00:00
17     1990-09-20 00:00:00
18     1976-11-18 00:00:00
19     2014-11-06 00:00:00
20     1995-12-15 00:00:00
21     1991-03-05 00:00:00
22     1977-10-20 00:00:00
23     1998-10-30 00:00:00
24     2000-10-03 00:00:00
25     2003-05-09 00:00:00
26     2003-04-18 00:00:00
27     1997-12-20 00:00:00
28     2019-11-07 00:00:00
29     1948-03-11 00:00:00
30     1955-08-19 00:00:00
31     2000-05-19 00:00:00
32     2006-10-27 00:00:00
33     2006-12-22 00:00:00
34     1985-10-18 00:00:00
35     1999-08-27 00:00:00
36     1995-04-07 00:00:00
37     1991-12-19 00:00:00
3

Check NaT/null values in conversion

In [71]:
x = df_checkpoint1['Release year'].isnull()
null_indices = df_checkpoint1.loc[:, 'Release year'][x].index
df.loc[null_indices, 'Release year']

70    The 6th of marzo, year 1951
83                     1984-02-34
84                     1976-13-24
Name: Release year, dtype: object

Fix index 70

In [72]:
df_checkpoint1.loc[70, 'Release year'] = pd.Timestamp(1951, 3, 6)
df_checkpoint1.loc[70, 'Release year']

Timestamp('1951-03-06 00:00:00')

Fix index 83 and 84 as 0 Timestamps

In [73]:
filler_release_year = pd.Timestamp.min

In [74]:
# df_checkpoint1.loc[83, 'Release year'] = filler_release_year
# df_checkpoint1.loc[84, 'Release year'] = filler_release_year
df_checkpoint1['Release year']

0      1995-02-10 00:00:00
1      1972-09-21 00:00:00
2      2008-07-23 00:00:00
3      1975-09-25 00:00:00
4      1994-10-28 00:00:00
5      2022-02-04 00:00:00
6      1994-03-11 00:00:00
7      1957-09-04 00:00:00
8      2010-09-24 00:00:00
9      1999-10-29 00:00:00
10     1994-10-06 00:00:00
11     2002-01-18 00:00:00
12     1966-12-23 00:00:00
14     1999-05-07 00:00:00
15     2003-01-16 00:00:00
16     1980-09-19 00:00:00
17     1990-09-20 00:00:00
18     1976-11-18 00:00:00
19     2014-11-06 00:00:00
20     1995-12-15 00:00:00
21     1991-03-05 00:00:00
22     1977-10-20 00:00:00
23     1998-10-30 00:00:00
24     2000-10-03 00:00:00
25     2003-05-09 00:00:00
26     2003-04-18 00:00:00
27     1997-12-20 00:00:00
28     2019-11-07 00:00:00
29     1948-03-11 00:00:00
30     1955-08-19 00:00:00
31     2000-05-19 00:00:00
32     2006-10-27 00:00:00
33     2006-12-22 00:00:00
34     1985-10-18 00:00:00
35     1999-08-27 00:00:00
36     1995-04-07 00:00:00
37     1991-12-19 00:00:00
3

In [75]:
df_checkpoint1.columns

Index(['Original Title', 'Release year', 'Genrë¨', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Cleaning 'Genre' column

Renaming the 'Genrë¨' column to Genre

In [76]:
df_checkpoint1 = df_checkpoint1.rename({'Genrë¨': 'Genre'}, axis=1)

Checking for any odd values

In [77]:
result = ''
for each in df_checkpoint1.loc[:, 'Genre'].unique():
    # log.debug(f'{each}')
    result += each + ', '
set(result.split(', ')[:-1])

{'Action',
 'Adventure',
 'Animation',
 'Biography',
 'Comedy',
 'Crime',
 'Drama',
 'Family',
 'Fantasy',
 'Film-Noir',
 'History',
 'Horror',
 'Music',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

In [78]:
df_checkpoint1.columns

Index(['Original Title', 'Release year', 'Genre', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Cleaning 'Duration' column

In [79]:
df_checkpoint1.loc[:, 'Duration']

0                 142
1                 175
2                 152
3                 220
4                    
5                 201
6                 Nan
7                  96
8                 148
9                 Inf
10                142
11               178c
12                161
14                NaN
15                179
16     Not Applicable
17                146
18                  -
19                169
20                127
21                118
22                121
23                169
24                189
25                130
26                125
27                116
28                132
29                130
30                207
31                155
32                151
33                130
34                116
35                119
36                110
37                137
38                106
39                 88
40                122
41                112
42                150
43                106
44                109
45                102
46        

Filler value for duration

In [80]:
filler_duration = 0

In [81]:
df_checkpoint1.loc[:, 'Duration'] = pd.to_numeric(df_checkpoint1['Duration'], errors='coerce', downcast='integer')

In [82]:
df_checkpoint1.loc[:, 'Duration'] = df_checkpoint1.loc[:, 'Duration'].infer_objects().fillna(filler_duration)
df_checkpoint1.loc[:, 'Duration']

0      142.0
1      175.0
2      152.0
3      220.0
4        0.0
5      201.0
6        0.0
7       96.0
8      148.0
9        inf
10     142.0
11       0.0
12     161.0
14       0.0
15     179.0
16       0.0
17     146.0
18       0.0
19     169.0
20     127.0
21     118.0
22     121.0
23     169.0
24     189.0
25     130.0
26     125.0
27     116.0
28     132.0
29     130.0
30     207.0
31     155.0
32     151.0
33     130.0
34     116.0
35     119.0
36     110.0
37     137.0
38     106.0
39      88.0
40     122.0
41     112.0
42     150.0
43     106.0
44     109.0
45     102.0
46     165.0
47      89.0
48     155.0
49      87.0
50     164.0
51     165.0
52     113.0
53      98.0
54     146.0
55     115.0
56     149.0
57     117.0
58     181.0
59     147.0
60     120.0
61      95.0
62     112.0
63     105.0
64     137.0
65     117.0
66     170.0
67     134.0
68     229.0
69     125.0
70     110.0
71     153.0
72     122.0
73     178.0
74     131.0
75      99.0
76     108.0
77      81.0

In [83]:
df_checkpoint1.loc[9, 'Duration'] = filler_duration
df_checkpoint1.loc[:, 'Duration'] = df_checkpoint1.loc[:, 'Duration'].astype(int)
df_checkpoint1.loc[:, 'Duration']

0      142
1      175
2      152
3      220
4        0
5      201
6        0
7       96
8      148
9        0
10     142
11       0
12     161
14       0
15     179
16       0
17     146
18       0
19     169
20     127
21     118
22     121
23     169
24     189
25     130
26     125
27     116
28     132
29     130
30     207
31     155
32     151
33     130
34     116
35     119
36     110
37     137
38     106
39      88
40     122
41     112
42     150
43     106
44     109
45     102
46     165
47      89
48     155
49      87
50     164
51     165
52     113
53      98
54     146
55     115
56     149
57     117
58     181
59     147
60     120
61      95
62     112
63     105
64     137
65     117
66     170
67     134
68     229
69     125
70     110
71     153
72     122
73     178
74     131
75      99
76     108
77      81
78     126
79     104
80     102
81     136
82     103
83     170
84     114
85     122
86     116
87     137
88     149
89     119
90     119
91     160

## Creating a backup

In [84]:
df_checkpoint2 = df_checkpoint1.copy()

## Cleaning 'Country' column

In [85]:
df_checkpoint2.loc[:, 'Country'].unique()

array(['USA', 'US', 'New Zealand', 'UK', 'New Zesland', 'Italy',
       'New Zeland', 'US.', 'Brazil', 'Japan', 'Italy1', 'South Korea',
       'France', 'Germany', 'India', 'Denmark', 'West Germany', 'Iran'],
      dtype=object)

Need to fix 'New Zealand',
 'New Zeland',
 'New Zesland', and 'US',
 'US.',
 'USA',

In [86]:
# x = df_checkpoint2.loc[:, 'Country'].replace(['New Zeland', 'New Zesland'], 'New Zealand')
replace_values = {
    'New Zeland': 'New Zealand',
    'New Zesland': 'New Zealand',
    'US': 'USA',
    'US.': 'USA'
}
df_checkpoint2.loc[:, 'Country'] = df_checkpoint2.loc[:, 'Country'].replace(replace_values)
df_checkpoint2.loc[:, 'Country']

0               USA
1               USA
2               USA
3               USA
4               USA
5       New Zealand
6               USA
7               USA
8               USA
9                UK
10              USA
11      New Zealand
12            Italy
14              USA
15      New Zealand
16              USA
17              USA
18              USA
19              USA
20              USA
21              USA
22              USA
23              USA
24              USA
25           Brazil
26            Japan
27           Italy1
28      South Korea
29              USA
30            Japan
31              USA
32              USA
33               UK
34              USA
35              USA
36           France
37              USA
38              USA
39              USA
40              USA
41           France
42               UK
43              USA
44              USA
45              USA
46            Italy
47            Japan
48            Italy
49              USA
50               UK


## Cleaning 'Content Rating' column

In [87]:
df_checkpoint2.loc[:, 'Content Rating'].unique()

array(['R', 'PG-13', 'Not Rated', 'Approved', 'PG', nan, 'Unrated', 'G'],
      dtype=object)

Filler variable for null 'Content Rating' values

In [88]:
filler_content_rating = 'Not Rated'

In [89]:
df_checkpoint2.loc[:, 'Content Rating'] = df_checkpoint2['Content Rating'].fillna(filler_content_rating)
df_checkpoint2.loc[:, 'Content Rating']

0              R
1              R
2          PG-13
3              R
4              R
5          PG-13
6              R
7      Not Rated
8          PG-13
9              R
10         PG-13
11         PG-13
12      Approved
14             R
15         PG-13
16            PG
17             R
18             R
19         PG-13
20             R
21             R
22            PG
23             R
24             R
25             R
26            PG
27     Not Rated
28     Not Rated
29            PG
30       Unrated
31             R
32             R
33         PG-13
34            PG
35             R
36     Not Rated
37             R
38             R
39             G
40     Not Rated
41     Not Rated
42             R
43             R
44             R
45            PG
46         PG-13
47     Not Rated
48     Not Rated
49             G
50         PG-13
51             R
52             R
53             G
54             R
55            PG
56     Not Rated
57             R
58     Not Rated
59            

In [90]:
df_checkpoint2.loc[:, 'Content Rating'].unique()

array(['R', 'PG-13', 'Not Rated', 'Approved', 'PG', 'Unrated', 'G'],
      dtype=object)

In [91]:
df_checkpoint2.columns

Index(['Original Title', 'Release year', 'Genre', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Cleaning 'Director' column

In [92]:
df_checkpoint2.loc[:, 'Director'].unique()

array(['Frank Darabont', 'Francis Ford Coppola', 'Christopher Nolan',
       'Quentin Tarantino', 'Peter Jackson', 'Steven Spielberg',
       'Sidney Lumet', 'David Fincher', 'Robert Zemeckis', 'Sergio Leone',
       'Lana Wachowski, Lilly Wachowski', 'Irvin Kershner',
       'Martin Scorsese', 'Milos Forman', 'Jonathan Demme',
       'George Lucas', 'Fernando Meirelles, KÃ¡tia Lund',
       'Hayao Miyazaki', 'Roberto Benigni', 'Bong Joon Ho', 'Frank Capra',
       'Akira Kurosawa', 'Ridley Scott', 'Tony Kaye', 'Luc Besson',
       'James Cameron', 'Bryan Singer', 'Roger Allers, Rob Minkoff',
       'Todd Phillips', 'Olivier Nakache, Ã‰ric Toledano',
       'Roman Polanski', 'Damien Chazelle', 'Alfred Hitchcock',
       'Michael Curtiz', 'Isao Takahata', 'Giuseppe Tornatore',
       'Charles Chaplin', 'Andrew Stanton', 'Stanley Kubrick',
       'Anthony Russo, Joe Russo', 'Chan-wook Park',
       'Lee Unkrich, Adrian Molina', 'Florian Henckel von Donnersmarck',
       'Bob Persichetti,

In [93]:
df_checkpoint2.columns

Index(['Original Title', 'Release year', 'Genre', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Cleaning 'Income' column

In [94]:
df_checkpoint2.loc[:, 'Income'] = df_checkpoint2.loc[:, 'Income'].str.replace('$ ', '')
df_checkpoint2.loc[:, 'Income']

0         28815245
1        246120974
2       1005455211
3      4o8,035,783
4        222831817
5       1142271098
6        322287794
7              576
8        869784991
9        101218804
10       678229452
11       887934303
12        25252481
14       465718588
15       951227416
16       549265501
17        46879633
18       108997629
19       696742056
20       327333559
21       272753884
22       775768912
23       482349603
24       286801374
25        30680793
26       355467056
27       230098753
28       257604912
29         6130720
30          322773
31       465361176
32       291465034
33       109676311
34       388774684
35        23875127
36        19552639
37       520884847
38        23341568
39       968511805
40      1074251311
41       426588510
42       120072577
43        48983260
44        32008644
45         4374761
46          112911
47          516962
48        13826605
49          457688
50      1081133191
51       425368238
52        39970386
53       521

Manually fix bad value

In [95]:
df_checkpoint2.loc[3, 'Income'] = 408035783

Cast to int

In [96]:
df_checkpoint2.loc[:, 'Income'] = df_checkpoint2.loc[:, 'Income'].astype(int)

In [97]:
df_checkpoint2.columns

Index(['Original Title', 'Release year', 'Genre', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', ' Votes ', 'Score'],
      dtype='object')

## Creating a backup

In [98]:
df_checkpoint3 = df_checkpoint2.copy()

## Cleaning 'Votes' column

In [99]:
df_checkpoint3.loc[:, ' Votes ']

0      2.278.845
1      1.572.674
2      2.241.615
3      1.098.714
4      1.780.147
5      1.604.280
6      1.183.248
7        668.473
8      2.002.816
9      1.807.440
10     1.755.490
11     1.619.920
12       672.499
14     1.632.315
15     1.449.778
16     1.132.073
17       991.505
18       891.071
19     1.449.256
20     1.402.015
21     1.234.134
22     1.204.107
23     1.203.825
24     1.112.336
25       685.856
26       626.693
27       605.648
28       470.931
29       388.310
30       307.958
31     1.308.193
32     1.159.703
33     1.155.723
34     1.027.330
35     1.014.218
36     1.007.598
37       974.970
38       968.947
39       917.248
40       855.097
41       736.691
42       707.942
43       690.732
44       586.765
45       509.953
46       295.220
47       225.438
48       223.050
49       211.250
50     1.480.582
51     1.317.856
52     1.098.879
53       974.734
54       869.480
55       865.510
56       796.486
57       768.874
58       754.786
59       591.2

Rename ' Votes ' column to 'Votes'

In [100]:
df_checkpoint3 = df_checkpoint3.rename({' Votes ': 'Votes'}, axis=1)

In [101]:
df_checkpoint3.loc[:, 'Votes']

0      2.278.845
1      1.572.674
2      2.241.615
3      1.098.714
4      1.780.147
5      1.604.280
6      1.183.248
7        668.473
8      2.002.816
9      1.807.440
10     1.755.490
11     1.619.920
12       672.499
14     1.632.315
15     1.449.778
16     1.132.073
17       991.505
18       891.071
19     1.449.256
20     1.402.015
21     1.234.134
22     1.204.107
23     1.203.825
24     1.112.336
25       685.856
26       626.693
27       605.648
28       470.931
29       388.310
30       307.958
31     1.308.193
32     1.159.703
33     1.155.723
34     1.027.330
35     1.014.218
36     1.007.598
37       974.970
38       968.947
39       917.248
40       855.097
41       736.691
42       707.942
43       690.732
44       586.765
45       509.953
46       295.220
47       225.438
48       223.050
49       211.250
50     1.480.582
51     1.317.856
52     1.098.879
53       974.734
54       869.480
55       865.510
56       796.486
57       768.874
58       754.786
59       591.2

## Cleaning 'Score' column

In [102]:
df_checkpoint3.loc[:, 'Score']

0         9.3
1         9.2
2          9.
3        9,.0
4        8,9f
5        08.9
6         8.9
7         8.9
8        8..8
9         8.8
10        8:8
11        8.8
12        8.8
14      ++8.7
15       8.7.
16     8,7e-0
17        8.7
18        8.7
19        8.6
20        8.6
21        8,6
22        8.6
23        8.6
24        8.6
25        8.6
26        8.6
27        8.6
28        8.6
29        8.6
30        8.6
31        8.5
32        8.5
33        8.5
34        8.5
35        8.5
36        8.5
37        8.4
38        8.4
39        8.4
40        8.4
41        8.4
42        8.4
43        8.4
44        8.3
45        8.3
46        8.3
47        8.3
48        8.3
49        8.3
50        8.3
51        8.3
52        8.2
53        8.2
54        8.2
55        8.2
56        8.2
57        8.2
58        8.2
59        8.2
60        8.1
61        8.1
62        8.1
63        8.1
64        8.1
65        8.1
66        8.1
67        8.0
68        8.0
69        8.0
70        8.0
71        8.0
72    

In [103]:
bad_values = {}
for idx, value in df_checkpoint3.loc[:, 'Score'].items():
    try:
        df_checkpoint3.loc[idx, 'Score'] = float(value)
    except ValueError:
        log.debug(f'{idx}: {value}')
        bad_values[idx] = value
bad_values

3: 9,.0
4: 8,9f
8: 8..8
10: 8:8
14: ++8.7
15: 8.7.
16: 8,7e-0
21: 8,6


{3: '9,.0',
 4: '8,9f',
 8: '8..8',
 10: '8:8',
 14: '++8.7',
 15: '8.7.',
 16: '8,7e-0',
 21: '8,6'}

In [104]:
replace_values = {
    ',.': '.',
    ',': '.',
    'f': '',
    '..': '.',
    ':': '.',
    '++': '',
}

# Strip values
df_checkpoint3.loc[bad_values.keys(), 'Score'] = df_checkpoint3.loc[bad_values.keys(), 'Score'].str.strip('++f.')
for to, value in replace_values.items():
    df_checkpoint3.loc[bad_values.keys(), 'Score'] = df_checkpoint3.loc[bad_values.keys(), 'Score'].str.replace(to, value)
df_checkpoint3.loc[:, 'Score'] = pd.to_numeric(df_checkpoint3.loc[:, 'Score'])
df_checkpoint3.loc[:, 'Score']

0      9.3
1      9.2
2      9.0
3      9.0
4      8.9
5      8.9
6      8.9
7      8.9
8      8.8
9      8.8
10     8.8
11     8.8
12     8.8
14     8.7
15     8.7
16     8.7
17     8.7
18     8.7
19     8.6
20     8.6
21     8.6
22     8.6
23     8.6
24     8.6
25     8.6
26     8.6
27     8.6
28     8.6
29     8.6
30     8.6
31     8.5
32     8.5
33     8.5
34     8.5
35     8.5
36     8.5
37     8.4
38     8.4
39     8.4
40     8.4
41     8.4
42     8.4
43     8.4
44     8.3
45     8.3
46     8.3
47     8.3
48     8.3
49     8.3
50     8.3
51     8.3
52     8.2
53     8.2
54     8.2
55     8.2
56     8.2
57     8.2
58     8.2
59     8.2
60     8.1
61     8.1
62     8.1
63     8.1
64     8.1
65     8.1
66     8.1
67     8.0
68     8.0
69     8.0
70     8.0
71     8.0
72     8.0
73     7.9
74     7.9
75     7.9
76     7.9
77     7.9
78     7.8
79     7.8
80     7.8
81     7.8
82     7.8
83     7.8
84     7.7
85     7.7
86     7.7
87     7.7
88     7.6
89     7.6
90     7.6
91     7.6

In [105]:
df_checkpoint3.columns

Index(['Original Title', 'Release year', 'Genre', 'Duration', 'Country',
       'Content Rating', 'Director', 'Income', 'Votes', 'Score'],
      dtype='object')

# Finished Cleaning Data

In [106]:
df_clean_data = df_checkpoint3.copy()
df_clean_data_styled = df_clean_data.style.format({'Income': '$'}, precision=2, thousands=',')
df_clean_data_styled

,Original Title,Release year,Genre,Duration,Country,Content Rating,Director,Income,Votes,Score
0,The Shawshank Redemption,1995-02-10 00:00:00,Drama,142,USA,R,Frank Darabont,$,2.278.845,9.30
1,The Godfather,1972-09-21 00:00:00,"Crime, Drama",175,USA,R,Francis Ford Coppola,$,1.572.674,9.20
2,The Dark Knight,2008-07-23 00:00:00,"Action, Crime, Drama",152,USA,PG-13,Christopher Nolan,$,2.241.615,9.00
3,The Godfather: Part II,1975-09-25 00:00:00,"Crime, Drama",220,USA,R,Francis Ford Coppola,$,1.098.714,9.00
4,Pulp Fiction,1994-10-28 00:00:00,"Crime, Drama",0,USA,R,Quentin Tarantino,$,1.780.147,8.90
5,The Lord of the Rings: The Return of the King,2022-02-04 00:00:00,"Action, Adventure, Drama",201,New Zealand,PG-13,Peter Jackson,$,1.604.280,8.90
6,Schindler's List,1994-03-11 00:00:00,"Biography, Drama, History",0,USA,R,Steven Spielberg,$,1.183.248,8.90
7,12 Angry Men,1957-09-04 00:00:00,"Crime, Drama",96,USA,Not Rated,Sidney Lumet,$,668.473,8.90
8,Inception,2010-09-24 00:00:00,"Action, Adventure, Sci-Fi",148,USA,PG-13,Christopher Nolan,$,2.002.816,8.80
9,Fight Club,1999-10-29 00:00:00,Drama,0,UK,R,David Fincher,$,1.807.440,8.80


## Export data

In [107]:
path_cleaned = library.path.get_cleaned_path(path)
df_clean_data.to_csv(path_cleaned)

In [108]:
df_clean_data

,Original Title,Release year,Genre,Duration,Country,Content Rating,Director,Income,Votes,Score
0,The Shawshank Redemption,1995-02-10 00:00:00,Drama,142,USA,R,Frank Darabont,28815245,2.278.845,9.3
1,The Godfather,1972-09-21 00:00:00,"Crime, Drama",175,USA,R,Francis Ford Coppola,246120974,1.572.674,9.2
2,The Dark Knight,2008-07-23 00:00:00,"Action, Crime, Drama",152,USA,PG-13,Christopher Nolan,1005455211,2.241.615,9.0
3,The Godfather: Part II,1975-09-25 00:00:00,"Crime, Drama",220,USA,R,Francis Ford Coppola,408035783,1.098.714,9.0
4,Pulp Fiction,1994-10-28 00:00:00,"Crime, Drama",0,USA,R,Quentin Tarantino,222831817,1.780.147,8.9
5,The Lord of the Rings: The Return of the King,2022-02-04 00:00:00,"Action, Adventure, Drama",201,New Zealand,PG-13,Peter Jackson,1142271098,1.604.280,8.9
6,Schindler's List,1994-03-11 00:00:00,"Biography, Drama, History",0,USA,R,Steven Spielberg,322287794,1.183.248,8.9
7,12 Angry Men,1957-09-04 00:00:00,"Crime, Drama",96,USA,Not Rated,Sidney Lumet,576,668.473,8.9
8,Inception,2010-09-24 00:00:00,"Action, Adventure, Sci-Fi",148,USA,PG-13,Christopher Nolan,869784991,2.002.816,8.8
9,Fight Club,1999-10-29 00:00:00,Drama,0,UK,R,David Fincher,101218804,1.807.440,8.8
